<a href="https://colab.research.google.com/github/angelrecalde2024/Power-System-Planning-and-Transmission-Design-2026/blob/main/INGP1118_TransmissionExpansion_SC_TEP_DC_OPF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

SC-TEP

Why is this problem harder than the TEP + SCOPF separately? In a non-secutiry constrained TEP, the topology is endogenous but you only enforce feasibility/limits in the base case.

In the SC-TEP,


*   You choose investments $y$ once (planning decision)
*   For each contingency (each single line outage), the system must still operate under DC constraints given the built network.
*   This means constraints are replicated over scenarios and depend on binary investment decisions.

Mathematically, it is a MILP with scenario replication.

The planning (investment) decision (global across all scenarios) are $y_{\ell}$ which is binary for each candidate circuit, and the corridor capacity $
\sum_{\ell \in \mathcal{C}(i, j)} y_{\ell} \leq N_{\max }
$. These are first-stage decision (same for base and all contingencies).

The operational decisions (scenario-specific) are: for each scenario $s$ (base + each N-1 outage), we find generator dispatch, bus angles, line flows, and consider a load shedding condition with $ENS_{i,s} \geq 0$. These are second-stage (recourse) decisions; they differ by scenario.

That is the hey conceptual jump: **corrective operation** per contingency, while investments are fixed.


THE PROBLEM WHEN THE CANDIDATE IS CONSIDERED IN THE SECURITY CONSTRAINT

For an existing line outage, it is always the scenario to include the line in the security constraint (N-1). However, for a candidate line outage, it is only 'possible' if that candidate were selected. This introduces the conditional scenario logic.

There are two standard treatment for the conditional scenario logic.


*   Treatment 1: Include all candidate outages, but make them inactive when not built. This can be implemented with linear constraints by 'switching off' the line's availability in that scenario using the same on/off logic as in base case.
*   Treatment 2: Build contingency set after investment. This is iterative/decomposition style (not needed for now).

Treatment 1 is selected in this case.

The constraint list is as follows.

For each scenario $s$,



1.   DC nodal balance (with ENS)

$$
\sum_{g \in i} P_{g, s}-P_{d, i} \text { LOAD_MULT }+E N S_{i, s}=\sum_{\ell \in \delta(i)} A_{i \ell} F_{\ell, s}
$$

2.   Generator limits
3.   Slack angle
4.   Existing line DC relation + limits (when not outaged in that scenario)
5.   Candidate line DC relation Big-M linking (base and scenarios)
6.   Candidate line capacity conditional with additional 'outaged in scenario' logic
7.   Parallel circuits per corridor

Plus the ENS budget across contingencies,

$$
\sum_{s \in \text { cont }} \sum_i E N S_{i, s} \leq E N S_{\max }
$$

CELL 1: Contingency feasibility exploration

Here the goal is to pre-identify contingencies that are structurally irrelevant or impossible given topology assumptions, to avoid “false infeasibility” and speed the SC-TEP MILP.

However, there is a conceptual catch: in SC-TEP, the set of feasible contingencies depends on $y$ because the network changes, so, the 'feasible contingencies' screening cannot be fully correct a priori unless you assume a reference investment state. Then, we screen only for structural islanding under the existing network (and optionally under 'all candidates built') to flag obviosly problematic outages.

This screening is conservative: it screens outages on the network with all candidates built.

*  If an outage islands even with all candidates built, it is structurally impossible under your candidate set → safe to drop.

*  If DC feasibility fails even with all candidates built and ENS=0, it’s also safe to drop (you can keep it to “prove infeasibility,” but it will block hard N-1).



In [30]:
# ============================================================
# CELL 1: Read Excel + build data + screen contingencies
# Reference network for screening: existing lines + ALL candidates built
# Output: SCENARIOS_KEPT, SCENARIOS_DROPPED, and core data structures
# ============================================================

!pip -q install pyomo highspy openpyxl pandas numpy

import math
import numpy as np
import pandas as pd
import pyomo.environ as pyo

# -----------------------
# USER KNOBS
# -----------------------
EXCEL = "/content/Data5busSystemSCTEP.xlsx"   # Colab: "/content/Data5busSystemTEP.xlsx"
BASE_MVA = 100.0
SLACK_BUS_EXT = 1

LOAD_MULT = 3.00     # <-- demand growth knob
THETA_MAX = math.pi  # for screening DC bounds
SCREEN_DC_FEASIBILITY = True   # True = run DC feasibility LP per outage (slower, but diagnostic)

# -----------------------
# Helpers: normalize headers
# -----------------------
def norm_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df

gen_df  = norm_cols(pd.read_excel(EXCEL, sheet_name="generator"))

# Force numeric generator data
for col in ["a","b","c","pmin","pmax"]:
    gen_df[col] = pd.to_numeric(gen_df[col], errors="coerce")

if gen_df[["a","b","c","pmin","pmax"]].isnull().any().any():
    raise ValueError("Generator sheet contains non-numeric values.")

load_df = norm_cols(pd.read_excel(EXCEL, sheet_name="load"))
line_df = norm_cols(pd.read_excel(EXCEL, sheet_name="lines"))
cand_df = norm_cols(pd.read_excel(EXCEL, sheet_name="candidates"))

req_gen  = {"generator bus","a","b","c","pmin","pmax"}
req_load = {"load bus","pload"}
req_line = {"from","to","x","flow limit"}
req_cand = {"from","to","x","flow limit","investment cost"}

for name, df, req in [
    ("generator", gen_df, req_gen),
    ("load", load_df, req_load),
    ("lines", line_df, req_line),
    ("candidates", cand_df, req_cand),
]:
    if not req.issubset(df.columns):
        raise ValueError(f"Sheet '{name}' missing columns. Required={sorted(req)} Found={list(df.columns)}")

# -----------------------
# Bus set (allow non-consecutive external labels)
# include candidate endpoints too
# -----------------------
buses_ext = sorted(
    set(gen_df["generator bus"].astype(int))
    | set(load_df["load bus"].astype(int))
    | set(line_df["from"].astype(int))
    | set(line_df["to"].astype(int))
    | set(cand_df["from"].astype(int))
    | set(cand_df["to"].astype(int))
)
if SLACK_BUS_EXT not in buses_ext:
    raise ValueError(f"Slack bus {SLACK_BUS_EXT} not in buses: {buses_ext}")

ext2int = {b:i for i,b in enumerate(buses_ext)}
int2ext = {i:b for b,i in ext2int.items()}
nb = len(buses_ext)
slack_i = ext2int[SLACK_BUS_EXT]

# -----------------------
# Load vector (MW), scaled by LOAD_MULT
# -----------------------
Pd = np.zeros(nb, dtype=float)
for _, r in load_df.iterrows():
    Pd[ext2int[int(r["load bus"])]] += float(r["pload"])
Pd *= float(LOAD_MULT)

# -----------------------
# Generators
# -----------------------
# Force numeric conversion from Excel
gen_df["a"] = pd.to_numeric(gen_df["a"], errors="coerce")
gen_df["b"] = pd.to_numeric(gen_df["b"], errors="coerce")
gen_df["c"] = pd.to_numeric(gen_df["c"], errors="coerce")
gen_df["pmin"] = pd.to_numeric(gen_df["pmin"], errors="coerce")
gen_df["pmax"] = pd.to_numeric(gen_df["pmax"], errors="coerce")

# Check if any value failed conversion
if gen_df[["a","b","c","pmin","pmax"]].isnull().any().any():
    print("Invalid numeric values detected in generator sheet:")
    print(gen_df)
    raise ValueError("Generator sheet contains non-numeric values.")

ng = len(gen_df)

gen_bus = np.zeros(ng, dtype=int)
Pmin = np.zeros(ng)
Pmax = np.zeros(ng)
a = np.zeros(ng)
b = np.zeros(ng)
c = np.zeros(ng)

for g, (_, r) in enumerate(gen_df.iterrows()):
    gen_bus[g] = ext2int[int(r["generator bus"])]
    a[g] = r["a"]
    b[g] = r["b"]
    c[g] = r["c"]
    Pmin[g] = r["pmin"]
    Pmax[g] = r["pmax"]

# -----------------------
# Existing + candidate branches
# We'll screen contingencies on the reference network where ALL candidates are built.
# -----------------------
existing = []
for _, r in line_df.iterrows():
    fi = ext2int[int(r["from"])]
    ti = ext2int[int(r["to"])]
    X  = float(r["x"])
    Fm = float(r["flow limit"])
    if abs(X) < 1e-9:
        raise ValueError("Near-zero X found in 'lines'.")
    existing.append((fi, ti, X, Fm))

candidates = []
for idx, r in cand_df.iterrows():
    fi = ext2int[int(r["from"])]
    ti = ext2int[int(r["to"])]
    X  = float(r["x"])
    Fm = float(r["flow limit"])
    inv = float(r["investment cost"])
    if abs(X) < 1e-9:
        raise ValueError("Near-zero X found in 'candidates'.")
    candidates.append((fi, ti, X, Fm, inv))

nl = len(existing)
nc = len(candidates)

print("=== Data summary (scaled load) ===")
print(f"buses={nb}, gens={ng}, existing lines={nl}, candidate lines={nc}")
print(f"Total load={Pd.sum():.3f} MW, Sum Pmax={Pmax.sum():.3f} MW, LOAD_MULT={LOAD_MULT}")
print(f"Slack ext={SLACK_BUS_EXT} -> int={slack_i}")

# -----------------------
# Build adjacency for reference network (existing + all candidates)
# -----------------------
def build_adj(remove_existing=None, remove_candidate=None):
    adj = {i:set() for i in range(nb)}
    for l in range(nl):
        if remove_existing is not None and l == remove_existing:
            continue
        u,v,_,_ = existing[l]
        adj[u].add(v); adj[v].add(u)
    for k in range(nc):
        if remove_candidate is not None and k == remove_candidate:
            continue
        u,v,_,_,_ = candidates[k]
        adj[u].add(v); adj[v].add(u)
    return adj

def is_connected_from_slack(adj):
    seen=set()
    stack=[slack_i]
    while stack:
        u=stack.pop()
        if u in seen:
            continue
        seen.add(u)
        stack.extend([v for v in adj[u] if v not in seen])
    return len(seen)==nb

# -----------------------
# Optional: DC feasibility LP under limits for the reference network with one outage
# Variables: theta, Pg, Fe, Fc
# Uses strict feasibility (no ENS) in screening.
# -----------------------
def dc_feasible_reference(remove_existing=None, remove_candidate=None):
    # connectivity first
    adj = build_adj(remove_existing, remove_candidate)
    if not is_connected_from_slack(adj):
        return (False, "islanding")

    # Pyomo feasibility LP
    m = pyo.ConcreteModel()
    m.B = pyo.RangeSet(0, nb-1)
    m.G = pyo.RangeSet(0, ng-1)
    m.LE = pyo.RangeSet(0, nl-1)
    m.LC = pyo.RangeSet(0, nc-1)

    m.theta = pyo.Var(m.B, bounds=(-THETA_MAX, THETA_MAX))
    m.theta[slack_i].fix(0.0)
    m.Pg = pyo.Var(m.G)

    # generator bounds
    def gb(mdl,g):
        return (float(Pmin[g]), mdl.Pg[g], float(Pmax[g]))
    m.gb = pyo.Constraint(m.G, rule=gb)

    # flows
    m.Fe = pyo.Var(m.LE)
    m.Fc = pyo.Var(m.LC)

    # existing DC + limits (skip outaged existing, enforce Fe=0)
    m.ex = pyo.ConstraintList()
    m.exlim = pyo.ConstraintList()
    for l in range(nl):
        fi,ti,X,Fm = existing[l]
        if remove_existing is not None and l==remove_existing:
            m.ex.add(m.Fe[l]==0.0)
        else:
            m.ex.add(m.Fe[l] == BASE_MVA*(m.theta[fi]-m.theta[ti])/float(X))
            m.exlim.add(m.Fe[l] <= float(Fm))
            m.exlim.add(-m.Fe[l] <= float(Fm))

    # candidate DC + limits in reference (ALL built), unless outaged => Fc=0
    m.ca = pyo.ConstraintList()
    m.calim = pyo.ConstraintList()
    for k in range(nc):
        fi,ti,X,Fm,inv = candidates[k]
        if remove_candidate is not None and k==remove_candidate:
            m.ca.add(m.Fc[k]==0.0)
        else:
            m.ca.add(m.Fc[k] == BASE_MVA*(m.theta[fi]-m.theta[ti])/float(X))
            m.calim.add(m.Fc[k] <= float(Fm))
            m.calim.add(-m.Fc[k] <= float(Fm))

    # incidence for balance
    out_e={i:[] for i in range(nb)}; in_e={i:[] for i in range(nb)}
    for l in range(nl):
        fi,ti,_,_ = existing[l]
        out_e[fi].append(l); in_e[ti].append(l)
    out_c={i:[] for i in range(nb)}; in_c={i:[] for i in range(nb)}
    for k in range(nc):
        fi,ti,_,_,_ = candidates[k]
        out_c[fi].append(k); in_c[ti].append(k)

    def bal(mdl,i):
        Pg_i = sum(mdl.Pg[g] for g in range(ng) if int(gen_bus[g])==i)
        net = sum(mdl.Fe[l] for l in out_e[i]) - sum(mdl.Fe[l] for l in in_e[i])
        net += sum(mdl.Fc[k] for k in out_c[i]) - sum(mdl.Fc[k] for k in in_c[i])
        return Pg_i - float(Pd[i]) == net
    m.balance = pyo.Constraint(m.B, rule=bal)

    m.obj = pyo.Objective(expr=0.0)

    solver = pyo.SolverFactory("highs")
    try:
        res = solver.solve(m, tee=False, load_solutions=False)
        term = str(res.solver.termination_condition).lower()
        if "optimal" in term or "feasible" in term:
            return (True, "feasible")
        if "infeasible" in term:
            return (False, "dc infeasible under limits")
        return (False, f"solver status={res.solver.status} term={res.solver.termination_condition}")
    except Exception as e:
        return (False, f"solver exception: {e}")

# -----------------------
# Build contingency list: all existing + all candidate outages
# Scenario IDs:
#   s=0 base
#   s=1..nl existing outage l = s-1
#   s=nl+1..nl+nc candidate outage k = s-(nl+1)
# -----------------------
SCENARIOS_KEPT = [0]
SCENARIOS_DROPPED = []  # list of dict rows
SCENARIO_META = {0: {"type":"base","out_kind":None,"out_id":None}}

# base connectivity check
if not is_connected_from_slack(build_adj()):
    raise RuntimeError("Reference network (existing+all candidates) is islanded. Check data.")

# existing outages
for l in range(nl):
    meta = {"type":"existing","out_kind":"existing","out_id":l}
    # structural check
    adj = build_adj(remove_existing=l)
    if not is_connected_from_slack(adj):
        SCENARIOS_DROPPED.append({**meta, "reason":"islanding (even with all candidates built)"})
        continue
    if SCREEN_DC_FEASIBILITY:
        ok, reason = dc_feasible_reference(remove_existing=l)
        if not ok:
            SCENARIOS_DROPPED.append({**meta, "reason":reason})
            continue
    s = len(SCENARIO_META)
    SCENARIO_META[s]=meta
    SCENARIOS_KEPT.append(s)

# candidate outages
for k in range(nc):
    meta = {"type":"candidate","out_kind":"candidate","out_id":k}
    adj = build_adj(remove_candidate=k)
    if not is_connected_from_slack(adj):
        SCENARIOS_DROPPED.append({**meta, "reason":"islanding (even with all candidates built)"})
        continue
    if SCREEN_DC_FEASIBILITY:
        ok, reason = dc_feasible_reference(remove_candidate=k)
        if not ok:
            SCENARIOS_DROPPED.append({**meta, "reason":reason})
            continue
    s = len(SCENARIO_META)
    SCENARIO_META[s]=meta
    SCENARIOS_KEPT.append(s)

print("\n=== Screening result (reference: all candidates built) ===")
print(f"Kept scenarios: {len(SCENARIOS_KEPT)} (includes base)")
print(f"Dropped scenarios: {len(SCENARIOS_DROPPED)}")

if SCENARIOS_DROPPED:
    df_drop = pd.DataFrame(SCENARIOS_DROPPED)
    # add human readable endpoints
    def endpoints(row):
        if row["out_kind"]=="existing":
            fi,ti,_,_ = existing[int(row["out_id"])]
        else:
            fi,ti,_,_,_ = candidates[int(row["out_id"])]
        return f"{int2ext[fi]}-{int2ext[ti]}"
    df_drop["corridor"] = df_drop.apply(endpoints, axis=1)
    display(df_drop)

# These objects are used by CELL 2:
# SCENARIOS_KEPT, SCENARIO_META, nb, ng, nl, nc, ext2int/int2ext, Pd, Pmin/Pmax, gen_bus, a/b/c,
# existing, candidates, slack_i, BASE_MVA, THETA_MAX

=== Data summary (scaled load) ===
buses=5, gens=2, existing lines=6, candidate lines=7
Total load=630.000 MW, Sum Pmax=1400.000 MW, LOAD_MULT=3.0
Slack ext=1 -> int=0

=== Screening result (reference: all candidates built) ===
Kept scenarios: 14 (includes base)
Dropped scenarios: 0


Cell 2: solve Security-Constrained TEP (SC-TEP) as a MILP with:

*  corrective dispatch per scenario,

*  base-case generation cost only (PWL approximation),

*  investment cost,

*  line limits in every scenario,

*  N-1 includes existing + candidate lines,

*  candidate contingencies are inactive if not built (no conditional scenario generation),

*  global ENS budget over all contingencies; ENS_MAX=0 ⇒ hard N-1.

It also includes a post-solve LP re-solve with binaries fixed to extract base-case LMPs (dual of base balance constraints), because duals are not defined for MILP.

NOTES:

*  Hard N-1 is $ENS_{MAX}=0$
*  Candidate contingencies are 'inactive' if not built because $F_c \leq y F^{max}$ already enforces zero flows, and the candidate-outage scenario forces $F_c=0$ (no extra restriction if $y=0$).

In [31]:
# ============================================================
# CELL 2: Security-Constrained TEP (SC-TEP) MILP
# - Corrective dispatch per scenario
# - Base-case gen cost only + investment cost
# - N-1 includes existing + candidate lines (candidate outages inactive if not built)
# - ENS allowed with global budget ENS_MAX (ENS_MAX=0 => hard N-1)
# - Corridor cap: <= MAX_PARALLEL_PER_CORRIDOR
# - LOAD_MULT already applied in Cell 1
# ============================================================

import math
import numpy as np
import pandas as pd
import pyomo.environ as pyo

# -----------------------
# USER KNOBS
# -----------------------
K_SEG = 5
MAX_PARALLEL_PER_CORRIDOR = 2

ENS_MAX = 100.0     # global ENS budget over all contingencies (MW); 0 => hard N-1
THETA_MAX = math.pi
M_MULT = 1.2

# -----------------------
# PWL gen cost (base case only)
# C(P)=a+bP+cP^2 approximated by K segments on [Pmin,Pmax]
# -----------------------
def pwl_params(Pmin_g, Pmax_g, ag, bg, cg, K):
    pts = np.linspace(Pmin_g, Pmax_g, K+1)
    dP = np.diff(pts)
    C = ag + bg*pts + cg*(pts**2)
    slope = np.diff(C) / dP
    return dP, slope

dP_seg = np.zeros((ng, K_SEG), dtype=float)
slope_seg = np.zeros((ng, K_SEG), dtype=float)
for g in range(ng):
    dP_seg[g,:], slope_seg[g,:] = pwl_params(Pmin[g], Pmax[g], a[g], b[g], c[g], K_SEG)

# -----------------------
# Corridor grouping for candidates (undirected corridor key)
# -----------------------
corridors = {}
for k in range(nc):
    fi,ti,X,Fm,inv = candidates[k]
    key = (min(fi,ti), max(fi,ti))
    corridors.setdefault(key, []).append(k)

# -----------------------
# Big-M for candidate on/off DC relation
# DC: F = BASE_MVA*(theta_f - theta_t)/X
# With theta bounds: |dtheta| <= 2*THETA_MAX
# Max |BASE_MVA*dtheta/X| <= BASE_MVA*(2*THETA_MAX)/|X|
# Choose M >= that + Fmax (moderate safety factor)
# -----------------------
def bigM(X, Fmax):
    return M_MULT * (abs(Fmax) + BASE_MVA*(2*THETA_MAX)/abs(X))

# -----------------------
# Scenario mapping from SCENARIO_META
# SCENARIOS_KEPT already includes base s=0.
# We'll build a compact scenario index set S = SCENARIOS_KEPT
# and an outage map: for each scenario s, out_kind/out_id
# -----------------------
S_list = list(SCENARIOS_KEPT)

def outage_of(s):
    mta = SCENARIO_META[s]
    return mta["out_kind"], mta["out_id"]  # (None,None) for base

# -----------------------
# Incidence lists for power balance
# We treat existing and candidate flows separately.
# Define "outflow - inflow" convention using directed from->to as given in data.
# -----------------------
out_e={i:[] for i in range(nb)}; in_e={i:[] for i in range(nb)}
for l in range(nl):
    fi,ti,_,_ = existing[l]
    out_e[fi].append(l); in_e[ti].append(l)

out_c={i:[] for i in range(nb)}; in_c={i:[] for i in range(nb)}
for k in range(nc):
    fi,ti,_,_,_ = candidates[k]
    out_c[fi].append(k); in_c[ti].append(k)

# -----------------------
# MODEL (MILP)
# -----------------------
m = pyo.ConcreteModel()

m.S = pyo.Set(initialize=S_list, ordered=True)
m.B = pyo.RangeSet(0, nb-1)
m.G = pyo.RangeSet(0, ng-1)
m.K = pyo.RangeSet(0, K_SEG-1)
m.LE = pyo.RangeSet(0, nl-1)
m.LC = pyo.RangeSet(0, nc-1)

# Variables
m.theta = pyo.Var(m.B, m.S, bounds=(-THETA_MAX, THETA_MAX))
for s in S_list:
    m.theta[slack_i, s].fix(0.0)

# Base-case PWL segments -> Pg_base
m.xseg0 = pyo.Var(m.G, m.K, domain=pyo.NonNegativeReals)
def seg_ub0(mdl,g,k):
    return mdl.xseg0[g,k] <= float(dP_seg[g,k])
m.seg_ub0 = pyo.Constraint(m.G, m.K, rule=seg_ub0)

def Pg0_expr(mdl,g):
    return float(Pmin[g]) + sum(mdl.xseg0[g,k] for k in mdl.K)
m.Pg0 = pyo.Expression(m.G, rule=Pg0_expr)

# Scenario dispatch Pg[g,s]
m.Pg = pyo.Var(m.G, m.S)

# Link base scenario dispatch to Pg0
def base_link(mdl,g):
    return mdl.Pg[g, 0] == mdl.Pg0[g]
m.base_link = pyo.Constraint(m.G, rule=base_link)

# Gen limits for all scenarios
def gen_bounds(mdl,g,s):
    return (float(Pmin[g]), mdl.Pg[g,s], float(Pmax[g]))
m.gen_bounds = pyo.Constraint(m.G, m.S, rule=gen_bounds)

# Flows per scenario
m.Fe = pyo.Var(m.LE, m.S)   # existing line flows
m.Fc = pyo.Var(m.LC, m.S)   # candidate line flows

# Investment binaries
m.y = pyo.Var(m.LC, domain=pyo.Binary)

# ENS per bus per scenario (allow in contingencies only; base fixed to 0)
m.ENS = pyo.Var(m.B, m.S, domain=pyo.NonNegativeReals)
m.ENS_base0 = pyo.ConstraintList()
for i in range(nb):
    m.ENS_base0.add(m.ENS[i, 0] == 0.0)

# Existing line DC relation + limits with outage handling
m.ex_dc = pyo.ConstraintList()
m.ex_lim = pyo.ConstraintList()
for s in S_list:
    out_kind, out_id = outage_of(s)
    for l in range(nl):
        fi,ti,X,Fm = existing[l]
        if out_kind=="existing" and out_id==l:
            m.ex_dc.add(m.Fe[l,s] == 0.0)
        else:
            m.ex_dc.add(m.Fe[l,s] == BASE_MVA*(m.theta[fi,s]-m.theta[ti,s]) / float(X))
            m.ex_lim.add(m.Fe[l,s] <= float(Fm))
            m.ex_lim.add(-m.Fe[l,s] <= float(Fm))

# Candidate line: on/off DC relation + conditional capacity + outage handling
m.cand_dc_ub = pyo.ConstraintList()
m.cand_dc_lb = pyo.ConstraintList()
m.cand_lim = pyo.ConstraintList()

for s in S_list:
    out_kind, out_id = outage_of(s)
    for k in range(nc):
        fi,ti,X,Fm,inv = candidates[k]

        # If this scenario is "outage of candidate k", force flow to zero.
        # If y[k]=0, it's already forced to zero by capacity; so the scenario is inactive.
        if out_kind=="candidate" and out_id==k:
            m.cand_lim.add(m.Fc[k,s] == 0.0)
            continue

        # Conditional capacity always applies
        m.cand_lim.add(m.Fc[k,s] <= float(Fm) * m.y[k])
        m.cand_lim.add(-m.Fc[k,s] <= float(Fm) * m.y[k])

        # Big-M linking of DC relation, active only if y=1
        M = bigM(X, Fm)
        expr = m.Fc[k,s] - BASE_MVA*(m.theta[fi,s]-m.theta[ti,s]) / float(X)
        m.cand_dc_ub.add(expr <= M*(1 - m.y[k]))
        m.cand_dc_lb.add(-expr <= M*(1 - m.y[k]))

# Corridor cap: <= N circuits per corridor
m.corr_set = pyo.Set(initialize=list(corridors.keys()), dimen=2)
def corr_cap(mdl,i,j):
    return sum(mdl.y[k] for k in corridors[(i,j)]) <= int(MAX_PARALLEL_PER_CORRIDOR)
m.corr_cap = pyo.Constraint(m.corr_set, rule=corr_cap)

# Power balance at each bus per scenario (with ENS)
def balance(mdl, i, s):
    Pg_i = sum(mdl.Pg[g,s] for g in range(ng) if int(gen_bus[g])==i)
    net = sum(mdl.Fe[l,s] for l in out_e[i]) - sum(mdl.Fe[l,s] for l in in_e[i])
    net += sum(mdl.Fc[k,s] for k in out_c[i]) - sum(mdl.Fc[k,s] for k in in_c[i])
    return Pg_i - float(Pd[i]) + mdl.ENS[i,s] == net
m.balance = pyo.Constraint(m.B, m.S, rule=balance)

# Global ENS budget across contingencies (exclude base)
def ens_budget(mdl):
    return sum(mdl.ENS[i,s] for s in S_list if s != 0 for i in range(nb)) <= float(ENS_MAX)
m.ens_budget = pyo.Constraint(rule=ens_budget)

# Objective: base-case PWL gen cost + investment cost
gen_cost_base = sum(float(slope_seg[g,k]) * m.xseg0[g,k] for g in range(ng) for k in range(K_SEG))
inv_cost = sum(float(candidates[k][4]) * m.y[k] for k in range(nc))
m.obj = pyo.Objective(expr=gen_cost_base + inv_cost, sense=pyo.minimize)

# -----------------------
# Solve MILP robustly
# -----------------------
solver = pyo.SolverFactory("highs")
res = solver.solve(m, tee=False, load_solutions=False)
term = str(res.solver.termination_condition).lower()
print("MILP Solver:", res.solver.status, res.solver.termination_condition)
if "optimal" not in term and "feasible" not in term:
    raise RuntimeError("SC-TEP MILP not solved (infeasible or solver error). Try ENS_MAX > 0 or check candidates/limits.")
m.solutions.load_from(res)

# -----------------------
# Report investment and base dispatch
# -----------------------
y = np.array([int(round(pyo.value(m.y[k]))) for k in range(nc)], dtype=int)
Pg_base = np.array([pyo.value(m.Pg[g,0]) for g in range(ng)], dtype=float)

inv_total = sum(candidates[k][4]*y[k] for k in range(nc))
quad_cost = float(np.sum(a + b*Pg_base + c*(Pg_base**2)))

print("\n=== Investment decisions ===")
rows=[]
for k in range(nc):
    fi,ti,X,Fm,inv = candidates[k]
    rows.append([k, int2ext[fi], int2ext[ti], X, Fm, inv, y[k]])
cand_report = pd.DataFrame(rows, columns=["CandID","From","To","X","Limit(MW)","InvCost","Build y"])
display(cand_report.sort_values(["Build y","CandID"], ascending=[False, True]))
print(f"Total investment cost = {inv_total:.3f}")

print("\n=== Base-case dispatch (MW) and quadratic-cost report ===")
grows=[]
for g in range(ng):
    grows.append([g, int2ext[int(gen_bus[g])], Pg_base[g], float(a[g] + b[g]*Pg_base[g] + c[g]*(Pg_base[g]**2))])
gen_report = pd.DataFrame(grows, columns=["Gen","Bus","Pg_base(MW)","QuadCost($/h)"])
display(gen_report)
print(f"Total quadratic generation cost (reported) = {quad_cost:.3f} $/h")

# ENS summary
ens_tot = float(sum(pyo.value(m.ENS[i,s]) for s in S_list if s!=0 for i in range(nb)))
print(f"\nTotal ENS used across contingencies = {ens_tot:.6f} MW (budget ENS_MAX={ENS_MAX})")

# Worst-contingency ENS
ens_by_s=[]
for s in S_list:
    if s==0:
        continue
    ens_s = float(sum(pyo.value(m.ENS[i,s]) for i in range(nb)))
    out_kind,out_id = outage_of(s)
    if out_kind=="existing":
        fi,ti,_,_ = existing[out_id]
    else:
        fi,ti,_,_,_ = candidates[out_id]
    ens_by_s.append([s,out_kind,out_id,f"{int2ext[fi]}-{int2ext[ti]}",ens_s])
ens_df = pd.DataFrame(ens_by_s, columns=["Scenario","OutKind","OutID","Outage","ENS(MW)"]).sort_values("ENS(MW)", ascending=False)
print("\n=== Worst-contingency ENS (top 10) ===")
display(ens_df.head(10))

# print which contigencies are binding
for s in S_list:
    flows = [abs(pyo.value(m.Fe[l,s])) for l in range(nl)]
    max_flow = max(flows)
    print(f"Scenario {s}: max existing flow = {max_flow:.2f}")

for s in S_list:
    ens = sum(pyo.value(m.ENS[i,s]) for i in range(nb))
    if ens > 1e-6:
        print("ENS in scenario", s, "=", ens)

for l in range(nl):
    if abs(pyo.value(m.Fe[l,0])) > 0.99*existing[l][3]:
        print("Line", l, "congested in base case")

# ------------------------------------------------
# Approximate LMPs from marginal generation cost
# ------------------------------------------------

# Base-case generator outputs
Pg_base = np.array([pyo.value(m.Pg[g,0]) for g in range(ng)])

# Marginal cost of each generator (derivative of quadratic cost)
MC = b + 2*c*Pg_base

# Identify marginal generator
# (generator not at min or max)
tol = 1e-4
marginal = []
for g in range(ng):
    if (Pg_base[g] > Pmin[g] + tol) and (Pg_base[g] < Pmax[g] - tol):
        marginal.append(g)

if len(marginal) == 0:
    print("No interior marginal generator found (price determined by congestion).")
else:
    ref_price = MC[marginal[0]]

# Assign LMP equal to reference marginal cost
LMP_base = np.full(nb, ref_price)

lmp_table = pd.DataFrame({
    "Bus":[int2ext[i] for i in range(nb)],
    "LMP_base($/MW)":LMP_base
})

print("\n=== Base-case LMPs (approximate from marginal generator) ===")
display(lmp_table)

MILP Solver: ok optimal

=== Investment decisions ===


,CandID,From,To,X,Limit(MW),InvCost,Build y
0,0,1,2,0.0845,175.0,3500000.0,1
3,3,3,4,0.0845,175.0,3500000.0,1
1,1,1,5,0.2112,175.0,8800000.0,0
2,2,2,4,0.2112,175.0,8800000.0,0
4,4,4,5,0.2112,175.0,8800000.0,0
5,5,2,3,0.0845,175.0,3500000.0,0
6,6,2,5,0.2112,175.0,8800000.0,0


Total investment cost = 7000000.000

=== Base-case dispatch (MW) and quadratic-cost report ===


,Gen,Bus,Pg_base(MW),QuadCost($/h)
0,0,1,134.165516,17843.997103
1,1,3,495.834484,2640.812364


Total quadratic generation cost (reported) = 20484.809 $/h

Total ENS used across contingencies = 100.000000 MW (budget ENS_MAX=100.0)

=== Worst-contingency ENS (top 10) ===


,Scenario,OutKind,OutID,Outage,ENS(MW)
1,2,existing,1,1-5,65.0
4,5,existing,4,4-5,35.0
0,1,existing,0,1-2,0.0
2,3,existing,2,2-4,0.0
3,4,existing,3,3-4,0.0
5,6,existing,5,2-3,0.0
6,7,candidate,0,1-2,0.0
7,8,candidate,1,1-5,0.0
8,9,candidate,2,2-4,0.0
9,10,candidate,3,3-4,0.0


Scenario 0: max existing flow = 175.00
Scenario 1: max existing flow = 152.36
Scenario 2: max existing flow = 175.00
Scenario 3: max existing flow = 175.00
Scenario 4: max existing flow = 171.22
Scenario 5: max existing flow = 175.00
Scenario 6: max existing flow = 175.00
Scenario 7: max existing flow = 175.00
Scenario 8: max existing flow = 175.00
Scenario 9: max existing flow = 175.00
Scenario 10: max existing flow = 175.00
Scenario 11: max existing flow = 175.00
Scenario 12: max existing flow = 175.00
Scenario 13: max existing flow = 175.00
ENS in scenario 2 = 64.99999999999994
ENS in scenario 5 = 35.0
Line 5 congested in base case

=== Base-case LMPs (approximate from marginal generator) ===


,Bus,LMP_base($/MW)
0,1,130.026833
1,2,130.026833
2,3,130.026833
3,4,130.026833
4,5,130.026833


Below is a port-solve diagnostic block showing:
*  flow on each line in each contingency
*  utilization (%)
*  which lines are congested
*  which contingencies bind
*  which candidate lines are built
*  which candidates provide security benefit

Binding contingencies define expansion decisions.

If binding contingencies are 3-4 and 2-5, then, candidate 2-4 is built, it certainly provides security benefits, meaning that, the new lines relieve those contingency bottlenecks.

In [32]:
# ============================================================
# SECURITY DIAGNOSTIC TABLE
# congestion of every line in every contingency
# binding contingencies
# candidate lines providing security benefit
# ============================================================

print("\n==============================================================")
print(" SECURITY CONSTRAINED TEP DIAGNOSTIC TABLE")
print(" Line congestion, binding contingencies, and security benefit")
print("==============================================================\n")

rows = []

# tolerance
flow_tol = 1e-3
cong_tol = 0.98

for s in S_list:

    if s == 0:
        scen_name = "BASE"
    else:
        out_kind, out_id = outage_of(s)

        if out_kind == "existing":
            fi,ti,_,_ = existing[out_id]
        else:
            fi,ti,_,_,_ = candidates[out_id]

        scen_name = f"OUT {int2ext[fi]}-{int2ext[ti]}"

    # existing lines
    for l in range(nl):

        fi,ti,X,Fmax = existing[l]
        flow = pyo.value(m.Fe[l,s])

        util = abs(flow)/Fmax if Fmax>0 else 0
        congested = util >= cong_tol

        rows.append([
            scen_name,
            "existing",
            f"{int2ext[fi]}-{int2ext[ti]}",
            flow,
            Fmax,
            util*100,
            congested
        ])

    # candidate lines
    for k in range(nc):

        fi,ti,X,Fmax,inv = candidates[k]
        flow = pyo.value(m.Fc[k,s])

        util = abs(flow)/Fmax if Fmax>0 else 0
        congested = util >= cong_tol
        built = int(round(pyo.value(m.y[k])))

        rows.append([
            scen_name,
            "candidate",
            f"{int2ext[fi]}-{int2ext[ti]}",
            flow,
            Fmax,
            util*100,
            congested
        ])


diag = pd.DataFrame(rows, columns=[
    "Scenario",
    "LineType",
    "Corridor",
    "Flow(MW)",
    "Limit(MW)",
    "Utilization(%)",
    "Congested"
])

display(diag)


# ------------------------------------------------
# Identify binding contingencies
# ------------------------------------------------

binding = diag[diag["Congested"] == True]["Scenario"].unique()

print("\nBinding contingencies (lines at limit):")
for b1 in binding:
    print("  ", b1)


# ------------------------------------------------
# Candidate lines providing security benefit
# ------------------------------------------------

print("\nCandidate lines built by the optimizer:")

for k in range(nc):

    if int(round(pyo.value(m.y[k]))) == 1:

        fi,ti,X,Fmax,inv = candidates[k]

        print(f"  Candidate {int2ext[fi]}-{int2ext[ti]} built")

        # check if it carries flow in some contingency
        useful = False

        for s in S_list:

            if abs(pyo.value(m.Fc[k,s])) > flow_tol:
                useful = True
                break

        if useful:
            print("     -> Provides security benefit (used in contingency flows)")
        else:
            print("     -> Built but not heavily used")


 SECURITY CONSTRAINED TEP DIAGNOSTIC TABLE
 Line congestion, binding contingencies, and security benefit



,Scenario,LineType,Corridor,Flow(MW),Limit(MW),Utilization(%),Congested
0,BASE,existing,1-2,14.582758,175.0,8.333005,False
1,BASE,existing,1-5,105.000000,175.0,60.000000,False
2,BASE,existing,2-4,-5.834484,175.0,3.333991,False
3,BASE,existing,3-4,160.417242,175.0,91.666995,False
4,BASE,existing,4-5,105.000000,175.0,60.000000,False
...,...,...,...,...,...,...,...
177,OUT 2-5,candidate,2-4,-0.000000,175.0,0.000000,False
178,OUT 2-5,candidate,3-4,90.577943,175.0,51.758825,False
179,OUT 2-5,candidate,4-5,-0.000000,175.0,0.000000,False
180,OUT 2-5,candidate,2-3,-0.000000,175.0,0.000000,False



Binding contingencies (lines at limit):
   BASE
   OUT 1-2
   OUT 1-5
   OUT 2-4
   OUT 3-4
   OUT 4-5
   OUT 2-3
   OUT 2-5

Candidate lines built by the optimizer:
  Candidate 1-2 built
     -> Provides security benefit (used in contingency flows)
  Candidate 3-4 built
     -> Provides security benefit (used in contingency flows)


Finally, below is a compact 'one page' SC-TEP results report. It summarizes the main planning insights in a structured way.

1.  investment decisions
2.  base-case dispatch
3.  worst contingency
4.  ENS distribution
5.  congestion map
6.  security benefit of candidate lines

This is very similar to the summary planners present in transmission expansion studies.

In [33]:
# ============================================================
# SC-TEP SUMMARY REPORT
# ============================================================

print("\n==============================================================")
print("            SECURITY CONSTRAINED TEP SUMMARY REPORT")
print("==============================================================\n")

# ------------------------------------------------
# 1. INVESTMENT DECISIONS
# ------------------------------------------------

print("---- INVESTMENT DECISIONS ----")

inv_rows = []

for k in range(nc):

    fi,ti,X,Fmax,inv = candidates[k]
    yk = int(round(pyo.value(m.y[k])))

    inv_rows.append([
        k,
        int2ext[fi],
        int2ext[ti],
        Fmax,
        inv,
        yk
    ])

inv_table = pd.DataFrame(inv_rows, columns=[
    "Candidate",
    "From",
    "To",
    "Limit(MW)",
    "InvestmentCost",
    "Built"
])

display(inv_table)

total_inv = sum(candidates[k][4] * int(round(pyo.value(m.y[k]))) for k in range(nc))

print("Total investment cost =", total_inv)


# --- HARD TYPE FIX (do this once before using a,b,c) ---
import numpy as np

a = np.asarray(a, dtype=float)
b = np.asarray(b, dtype=float)
c = np.asarray(c, dtype=float)

# optional sanity prints (keep while debugging)
print("DEBUG dtypes:", a.dtype, b.dtype, c.dtype)
print("DEBUG first a,b,c:", a[:min(5,len(a))], b[:min(5,len(b))], c[:min(5,len(c))])

# ------------------------------------------------
# 2. BASE CASE GENERATION DISPATCH
# ------------------------------------------------

print("\n---- BASE CASE GENERATION ----")

Pg_base = np.array([pyo.value(m.Pg[g,0]) for g in range(ng)])

gen_rows = []

for g in range(ng):

    Pg = Pg_base[g]

    cost = a[g] + b[g]*Pg + c[g]*(Pg**2)

    gen_rows.append([
        g,
        int2ext[int(gen_bus[g])],
        Pg,
        cost
    ])

gen_table = pd.DataFrame(gen_rows, columns=[
    "Generator",
    "Bus",
    "Pg(MW)",
    "Cost"
])

display(gen_table)

print("Total generation =", Pg_base.sum())


# ------------------------------------------------
# 3. WORST CONTINGENCY
# ------------------------------------------------

print("\n---- WORST CONTINGENCY ----")

worst_util = 0
worst_row = None

for s in S_list:

    if s == 0:
        continue

    for l in range(nl):

        fi,ti,X,Fmax = existing[l]

        flow = abs(pyo.value(m.Fe[l,s]))

        util = flow/Fmax

        if util > worst_util:

            worst_util = util
            worst_row = ("existing", int2ext[fi], int2ext[ti], s, util)

print("Worst stress:", worst_row)


# ------------------------------------------------
# 4. ENS DISTRIBUTION
# ------------------------------------------------

print("\n---- LOAD SHEDDING (ENS) ----")

ens_rows = []

for s in S_list:

    if s == 0:
        continue

    ens = sum(pyo.value(m.ENS[i,s]) for i in range(nb))

    if ens > 1e-6:

        out_kind,out_id = outage_of(s)

        if out_kind == "existing":
            fi,ti,_,_ = existing[out_id]
        else:
            fi,ti,_,_,_ = candidates[out_id]

        ens_rows.append([
            s,
            f"{int2ext[fi]}-{int2ext[ti]}",
            ens
        ])

if len(ens_rows) == 0:

    print("No load shedding required (hard N-1 satisfied)")

else:

    ens_table = pd.DataFrame(ens_rows, columns=[
        "Scenario",
        "Outage",
        "ENS(MW)"
    ])

    display(ens_table)


# ------------------------------------------------
# 5. CONGESTION MAP (BASE CASE)
# ------------------------------------------------

print("\n---- BASE CASE CONGESTION ----")

cong_rows = []

for l in range(nl):

    fi,ti,X,Fmax = existing[l]

    flow = pyo.value(m.Fe[l,0])
    util = abs(flow)/Fmax

    cong_rows.append([
        int2ext[fi],
        int2ext[ti],
        flow,
        Fmax,
        util
    ])

cong_table = pd.DataFrame(cong_rows, columns=[
    "From",
    "To",
    "Flow",
    "Limit",
    "Utilization"
])

display(cong_table.sort_values("Utilization", ascending=False))


# ------------------------------------------------
# 6. SECURITY BENEFIT OF CANDIDATE LINES
# ------------------------------------------------

print("\n---- SECURITY BENEFIT OF BUILT CANDIDATES ----")

for k in range(nc):

    if int(round(pyo.value(m.y[k]))) == 1:

        fi,ti,X,Fmax,inv = candidates[k]

        used = False

        for s in S_list:

            if abs(pyo.value(m.Fc[k,s])) > 1e-3:
                used = True
                break

        print(f"Candidate {int2ext[fi]}-{int2ext[ti]} built")

        if used:
            print("   -> Carries contingency flow (security benefit)")
        else:
            print("   -> Built but rarely used")


            SECURITY CONSTRAINED TEP SUMMARY REPORT

---- INVESTMENT DECISIONS ----


,Candidate,From,To,Limit(MW),InvestmentCost,Built
0,0,1,2,175.0,3500000.0,1
1,1,1,5,175.0,8800000.0,0
2,2,2,4,175.0,8800000.0,0
3,3,3,4,175.0,3500000.0,1
4,4,4,5,175.0,8800000.0,0
5,5,2,3,175.0,3500000.0,0
6,6,2,5,175.0,8800000.0,0


Total investment cost = 7000000.0
DEBUG dtypes: float64 float64 float64
DEBUG first a,b,c: [400.68 395.37] [130.      4.423] [0.0001   0.000213]

---- BASE CASE GENERATION ----


,Generator,Bus,Pg(MW),Cost
0,0,1,134.165516,17843.997103
1,1,3,495.834484,2640.812364


Total generation = 630.0000000000001

---- WORST CONTINGENCY ----
Worst stress: ('existing', 3, 4, 2, 1.0)

---- LOAD SHEDDING (ENS) ----


,Scenario,Outage,ENS(MW)
0,2,1-5,65.0
1,5,4-5,35.0



---- BASE CASE CONGESTION ----


,From,To,Flow,Limit,Utilization
5,2,3,-175.000000,175.0,1.00000
3,3,4,160.417242,175.0,0.91667
4,4,5,105.000000,175.0,0.60000
1,1,5,105.000000,175.0,0.60000
0,1,2,14.582758,175.0,0.08333
2,2,4,-5.834484,175.0,0.03334



---- SECURITY BENEFIT OF BUILT CANDIDATES ----
Candidate 1-2 built
   -> Carries contingency flow (security benefit)
Candidate 3-4 built
   -> Carries contingency flow (security benefit)
